# 01 — Data Exploration

**Milestone M0.** Before writing any preprocessing, we look at the raw recordings
to understand their structure. The smartphone app writes **one flat CSV per run**,
where every row belongs to one of several data *streams* identified by the `source`
column. The goal here is to see what streams exist, how the sensors are stored, and
which data-quality issues we must handle later.

This notebook reads the raw CSVs directly (preprocessing comes in notebook 02).

In [1]:
import pandas as pd

RAW = "../data/raw"

# Load all four raw recordings; explore Run 1 in detail.
raw = {run_id: pd.read_csv("%s/Record_data_path_%d.csv" % (RAW, run_id))
       for run_id in [1, 2, 3, 4]}
df1 = raw[1]
print("Run 1 raw shape:", df1.shape)

Run 1 raw shape: (127472, 11)


## Columns

Every row shares the same wide schema; which columns are filled depends on the
stream. IMU rows fill `x, y, z`; BLE rows fill `address, rssi`; the `notes` column
carries the beacon name for scanned devices.

In [2]:
df1.head()

,timestamp_ms,timestamp_iso,source,id,cardinality,address,rssi,x,y,z,notes
0,1782119624720,2026-06-22T11:13:44.720+0200,beacon,NaN,-1.0,E9:C4:63:D1:6D:A2,-69.0,NaN,NaN,NaN,protocol=; name=arrive_emi8; serviceData={0000...
1,1782119624735,2026-06-22T11:13:44.735+0200,beacon,NaN,-1.0,16:7C:75:72:D6:BC,-92.0,NaN,NaN,NaN,protocol=; name=; serviceData=; manufacturerDa...
2,1782119624738,2026-06-22T11:13:44.738+0200,beacon,NaN,-1.0,02:9A:27:A9:87:68,-65.0,NaN,NaN,NaN,protocol=; name=; serviceData=; manufacturerDa...
3,1782119624754,2026-06-22T11:13:44.754+0200,beacon,NaN,-1.0,F0:CE:D3:B4:8E:42,-90.0,NaN,NaN,NaN,protocol=; name=arrive_emi10; serviceData={000...
4,1782119624768,2026-06-22T11:13:44.768+0200,beacon,NaN,-1.0,AC:37:43:88:19:33,-73.0,NaN,NaN,NaN,protocol=; name=HTC BS 339538; serviceData=; m...


In [3]:
print(list(df1.columns))

['timestamp_ms', 'timestamp_iso', 'source', 'id', 'cardinality', 'address', 'rssi', 'x', 'y', 'z', 'notes']


## The `source` column — several streams in one file

Each row is one of: `imu` (motion samples), `ble_rssi` (clean, beacon-mapped RSSI
readings we use for positioning), `beacon` (the raw promiscuous scan of every
nearby BLE device), and `live_ble_snapshot` (occasional full snapshots). The IMU
dominates by count because it is sampled at a high rate.

In [4]:
df1['source'].value_counts()

source
imu                  119012
beacon                 5789
ble_rssi               2664
live_ble_snapshot         7
Name: count, dtype: int64

## IMU sub-streams (the `id` column)

All IMU rows share `source == "imu"` and are told apart by `id`: `accel` (used for
step detection), `gyro` (used for heading), `mag` (magnetometer, unreliable
indoors), and `imu_processed` (the app's already-filtered IMU, kept only for
comparison).

In [5]:
df1[df1['source'] == 'imu']['id'].value_counts()

id
imu_processed    59502
accel            19868
gyro             19866
mag              19776
Name: count, dtype: int64

## BLE beacons seen

For `ble_rssi` rows the `id` is the beacon name. The installed project beacons are
named `arrive_emi*`; the scan also picks up unrelated devices, which we will drop
in preprocessing. Only **6 of the 8** installed beacons appear (the east-wing pair
is not covered by these paths).

In [6]:
ble_ids = df1[df1['source'] == 'ble_rssi']['id']
project = ble_ids[ble_ids.str.startswith('arrive_emi', na=False)]
print("distinct project beacons in Run 1:", sorted(project.unique()))
project.value_counts()

distinct project beacons in Run 1: ['arrive_emi1', 'arrive_emi10', 'arrive_emi2', 'arrive_emi3', 'arrive_emi4', 'arrive_emi8']


id
arrive_emi10    591
arrive_emi4     579
arrive_emi2     381
arrive_emi3     376
arrive_emi1     359
arrive_emi8     297
Name: count, dtype: int64

## Stream counts across all four runs

A quick cross-run view of how many rows each stream contributes.

In [7]:
summary = pd.DataFrame({run_id: df['source'].value_counts()
                        for run_id, df in raw.items()}).fillna(0).astype(int)
summary.index.name = 'source'
summary

,1,2,3,4
source,,,,
imu,119012,114620,152004,137112
beacon,5789,3657,4665,6055
ble_rssi,2664,2084,2487,2345
live_ble_snapshot,7,7,7,7


## Data-quality observations

- **Run 2 was originally a duplicate of Run 1** (byte-identical file); it has since
  been replaced with the correct recording. Preprocessing verifies this with a file
  hash (`is_duplicate`).
- **Unrelated BLE devices** appear in the raw scan (`source == beacon`, and some
  non-`arrive_emi` entries in `ble_rssi`); these are filtered out in preprocessing.
- **BLE is event-driven and irregular**, with occasional multi-second gaps, whereas
  the IMU is near-continuous — so the two streams are kept separate on a shared
  clock rather than merged.
- The **magnetometer** is recorded but not used for heading (indoor magnetic fields
  are unreliable); heading comes from the gyroscope instead.

Next: **02 — Preprocessing** turns these raw files into clean `Run` objects.